In [ ]:
import numpy as np
import json
import tensorflow as tf
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Input, GRU, Dense, Embedding
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint
from tensorflow.keras.optimizers import Adam

In [ ]:
# ── Load dataset from splits.json ─────────────────────────────────────────────

with open('splits.json', 'r', encoding='utf-8') as f:
    splits = json.load(f)

train_pairs = splits['train']
val_pairs   = splits['val']
test_pairs  = splits['test']

all_pairs     = train_pairs + val_pairs + test_pairs
english_words = [p[0].lower().strip() for p in all_pairs]
arabic_words  = [p[1] for p in all_pairs]

print(f"Train: {len(train_pairs)} | Val: {len(val_pairs)} | Test: {len(test_pairs)}")
print("Sample:", train_pairs[:3])

In [ ]:
# ── Vocabulary ────────────────────────────────────────────────────────────────

PAD   = '<PAD>'
START = '<START>'
END   = '<END>'

def build_vocab(words):
    chars = sorted(set(ch for w in words for ch in w))
    vocab = [PAD, START, END] + chars
    w2i   = {c: i for i, c in enumerate(vocab)}
    i2w   = {i: c for c, i in w2i.items()}
    return vocab, w2i, i2w

eng_vocab, eng_w2i, eng_i2w = build_vocab(english_words)
ara_vocab, ara_w2i, ara_i2w = build_vocab(arabic_words)

ENG_VOCAB_SIZE = len(eng_vocab)
ARA_VOCAB_SIZE = len(ara_vocab)
ENC_MAX_LEN    = max(len(w) for w in english_words)
DEC_MAX_LEN    = max(len(w) for w in arabic_words) + 2

print(f"English vocab: {ENG_VOCAB_SIZE} | max len: {ENC_MAX_LEN}")
print(f"Arabic  vocab: {ARA_VOCAB_SIZE} | max len: {DEC_MAX_LEN}")

In [ ]:
# ── Encode sequences ──────────────────────────────────────────────────────────

def encode_eng(word):
    ids = [eng_w2i.get(ch, 0) for ch in word]
    ids += [eng_w2i[PAD]] * (ENC_MAX_LEN - len(ids))
    return ids

def encode_ara(word):
    ids = [ara_w2i[START]] + [ara_w2i.get(ch, 0) for ch in word] + [ara_w2i[END]]
    ids += [ara_w2i[PAD]] * (DEC_MAX_LEN - len(ids))
    return ids

encoder_inputs     = np.array([encode_eng(w) for w in english_words])
dec_full           = np.array([encode_ara(w) for w in arabic_words])
decoder_inputs     = dec_full[:, :-1]
decoder_targets    = dec_full[:, 1:]
decoder_targets_oh = tf.keras.utils.to_categorical(decoder_targets, num_classes=ARA_VOCAB_SIZE)

print("encoder_inputs :", encoder_inputs.shape)
print("decoder_inputs :", decoder_inputs.shape)
print("decoder_targets:", decoder_targets_oh.shape)

In [ ]:
# ── Model ─────────────────────────────────────────────────────────────────────
# GRU only has state_h (no state_c like LSTM)

EMBED_DIM  = 64
LATENT_DIM = 256

# Encoder
enc_input  = Input(shape=(ENC_MAX_LEN,), name='enc_input')
enc_embed  = Embedding(ENG_VOCAB_SIZE, EMBED_DIM, mask_zero=True, name='enc_embedding')(enc_input)
_, state_h = GRU(LATENT_DIM, return_state=True, name='enc_gru')(enc_embed)

# Decoder
dec_input  = Input(shape=(DEC_MAX_LEN - 1,), name='dec_input')
dec_embed  = Embedding(ARA_VOCAB_SIZE, EMBED_DIM, mask_zero=True, name='dec_embedding')(dec_input)
dec_gru    = GRU(LATENT_DIM, return_sequences=True, return_state=True, name='dec_gru')
dec_out, _ = dec_gru(dec_embed, initial_state=state_h)
dec_dense  = Dense(ARA_VOCAB_SIZE, activation='softmax', name='dec_output')(dec_out)

model = Model([enc_input, dec_input], dec_dense)
model.compile(optimizer=Adam(1e-3), loss='categorical_crossentropy', metrics=['accuracy'])
model.summary()

In [ ]:
# ── Train ─────────────────────────────────────────────────────────────────────

early_stop = EarlyStopping(monitor='val_loss', patience=10, restore_best_weights=True)
checkpoint = ModelCheckpoint('gru_seq2seq.h5', monitor='val_loss', save_best_only=True)

history = model.fit(
    [encoder_inputs, decoder_inputs],
    decoder_targets_oh,
    batch_size=64,
    epochs=100,
    validation_split=0.1,
    callbacks=[early_stop, checkpoint]
)

model.save('gru_seq2seq.h5')
print('Saved → gru_seq2seq.h5')

In [ ]:
# ── Inference models ──────────────────────────────────────────────────────────

# Encoder: English input → hidden state
inf_enc_model = Model(enc_input, state_h)

# Decoder: 1 char + state → next char + new state
inf_h_in   = Input(shape=(LATENT_DIM,), name='inf_h')
inf_dec_in = Input(shape=(1,),          name='inf_dec_in')

inf_embed      = model.get_layer('dec_embedding')(inf_dec_in)
inf_out, inf_h = model.get_layer('dec_gru')(inf_embed, initial_state=inf_h_in)
inf_dense_out  = model.get_layer('dec_output')(inf_out)

inf_dec_model = Model(
    [inf_dec_in, inf_h_in],
    [inf_dense_out, inf_h]
)

In [ ]:
# ── transliterate() ───────────────────────────────────────────────────────────

def transliterate(english_word):
    """
    Takes an English word and returns its Arabic transliteration.
    Used by the main pipeline after the CNN predicts letters.

    Example:
        transliterate('ahmed')    → 'احمد'
        transliterate('computer') → 'كمبيوتر'
    """
    word = english_word.lower().strip()

    ids = [eng_w2i.get(ch, 0) for ch in word]
    ids += [eng_w2i[PAD]] * (ENC_MAX_LEN - len(ids))
    enc_in = np.array([ids])

    # GRU returns single state (not h and c like LSTM)
    state = inf_enc_model.predict(enc_in, verbose=0)

    target = np.array([[ara_w2i[START]]])
    result = []

    for _ in range(30):
        out, state = inf_dec_model.predict([target, state], verbose=0)
        pred_id    = np.argmax(out[0, 0, :])
        pred_char  = ara_i2w[pred_id]

        if pred_char in (END, PAD):
            break

        result.append(pred_char)
        target = np.array([[pred_id]])

    return ''.join(result)


# ── Test on test split ────────────────────────────────────────────────────────
print(f"{'English':20s} {'Predicted':20s} {'Ground Truth'}")
print('-' * 60)
for eng, ara_gt in test_pairs[:10]:
    pred = transliterate(eng)
    print(f"{eng:20s} {pred:20s} {ara_gt}")